<a href="https://colab.research.google.com/github/wllamjp/NSDC-NBA-Analysis/blob/main/Eras%20%26%20Pace/q1_matchup_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
import pandas as pd

url = "https://raw.githubusercontent.com/fivethirtyeight/data/master/nba-elo/nbaallelo.csv"
nba = pd.read_csv(url)

print(nba.head())

   gameorder       game_id lg_id  _iscopy  year_id  date_game  seasongame  \
0          1  194611010TRH   NBA        0     1947  11/1/1946           1   
1          1  194611010TRH   NBA        1     1947  11/1/1946           1   
2          2  194611020CHS   NBA        0     1947  11/2/1946           1   
3          2  194611020CHS   NBA        1     1947  11/2/1946           2   
4          3  194611020DTF   NBA        0     1947  11/2/1946           1   

   is_playoffs team_id  fran_id  ...  win_equiv  opp_id  opp_fran  opp_pts  \
0            0     TRH  Huskies  ...  40.294830     NYK    Knicks       68   
1            0     NYK   Knicks  ...  41.705170     TRH   Huskies       66   
2            0     CHS    Stags  ...  42.012257     NYK    Knicks       47   
3            0     NYK   Knicks  ...  40.692783     CHS     Stags       63   
4            0     DTF  Falcons  ...  38.864048     WSC  Capitols       50   

   opp_elo_i  opp_elo_n  game_location  game_result  forecast notes 

In [24]:
print(nba.shape)
print(nba.columns)

# league ID (either NBA or ABA)
print(nba["lg_id"].value_counts())

(126314, 23)
Index(['gameorder', 'game_id', 'lg_id', '_iscopy', 'year_id', 'date_game',
       'seasongame', 'is_playoffs', 'team_id', 'fran_id', 'pts', 'elo_i',
       'elo_n', 'win_equiv', 'opp_id', 'opp_fran', 'opp_pts', 'opp_elo_i',
       'opp_elo_n', 'game_location', 'game_result', 'forecast', 'notes'],
      dtype='object')
lg_id
NBA    118016
ABA      8298
Name: count, dtype: int64


In [25]:
# keeping only NBA games by removing ABA games
nba = nba[nba["lg_id"] == "NBA"]

# removes playoff games
nba = nba[nba["is_playoffs"] == 0]

# keeps 1979-1980 season and onward b/c the 3 point line was added that year
nba = nba[nba["year_id"] >= 1980]

# removes 19 neutral games that are neither home or away
print(nba["game_location"].value_counts())
nba = nba[nba["game_location"] != "N"]

print(nba.shape)

game_location
A    39620
H    39620
N       38
Name: count, dtype: int64
(79240, 23)


In [26]:
# remove duplicate games (b/c every game is listed twice)
nba = nba[nba["fran_id"] < nba["opp_fran"]]

print(nba.shape)
print(nba["game_id"].duplicated().sum())

# keeps necessary columns including franchise_id so that relocated/renamed
# teams aren't lost
nba = nba[["game_id", "year_id", "fran_id", "opp_fran", "game_location", "pts", "opp_pts", "forecast"]].copy()

# add variables (win = 1 if team won)
nba["win"] = (nba["pts"] > nba["opp_pts"]).astype(int)

# creates edge = win minus forecast
# need to take Brandon's raw data later for league-average adjustments
nba["edge"] = nba["win"] - nba["forecast"]

# era = time period based on rule changes (2005 = first season after
# hand-check ban)
def era(season):
  # 1994-1995 marked the first shortened 3-point line season
  # now all eras are of a more similar size
  if season <= 1994:
    return "1980-1994"
  elif season <= 2004:
    return "1995-2004"
  else:
    return "2005-2015"

nba["era"] = nba["year_id"].apply(era)
print(nba.head())

(39620, 23)
0
            game_id  year_id    fran_id opp_fran game_location  pts  opp_pts  \
41743  197910120ATL     1980      Hawks   Knicks             H  121      104   
41744  197910120BOS     1980    Celtics  Rockets             H  114      106   
41747  197910120DET     1980     Pacers  Pistons             A  105      114   
41749  197910120KCK     1980      Bucks    Kings             A  103      105   
41750  197910120NJN     1980  Cavaliers     Nets             A   95      102   

       forecast  win      edge        era  
41743  0.800678    1  0.199322  1980-1994  
41744  0.449544    1  0.550456  1980-1994  
41747  0.423580    0 -0.423580  1980-1994  
41749  0.395580    0 -0.395580  1980-1994  
41750  0.362611    0 -0.362611  1980-1994  


In [27]:
nba.to_csv("q1_games_clean.csv", index=False)